# Session 1: from a catalogue to a defensible result

Whether you work with survey catalogues or simulation snapshots, a defensible analysis keeps columns aligned, tracks physical units from provenance, tests assumptions explicitly, and separates data-quality cuts from physical selection effects.

Our 400-row galaxy table is **synthetic teaching data**: input distances $d$ are exact, and recession velocities follow an injected linear relation ($v = 70\,d + 300\text{ km/s}$) with independent Gaussian measurement noise. It is not a cosmological measurement.

## Environments and kernels: pip, conda and uv

Research projects routinely need conflicting Python or library versions; even on CPD, the general `python3` kernel runs Conda Python 3.12 whereas **Python PhD (locked)** runs an isolated Python 3.13 `.venv` at `/opt/course/.venv`. **Never `pip install` into system Python or Conda `(base)`** — isolate each project.

| Tool | What it manages | Spec / lock files | Typical research fit |
| --- | --- | --- | --- |
| **`pip` + `venv`** | Built-in `.venv` + PyPI wheels | `pyproject.toml`, `requirements.txt` | Lightweight pure-Python or wheel-only packages |
| **`uv`** *(used here)* | Python interpreter + `.venv` + fast wheel resolver | `pyproject.toml` + hash-checked `uv.lock` (`uv sync --frozen`) | Fast, bit-reproducible environments across machines |
| **`conda` / `mamba`** | Python **plus compiled C/Fortran/CUDA libraries** (`conda-forge`) | `environment.yml` (`conda-lock.yml`) | Stacks needing binary astronomy libraries (`CFITSIO`, `FFTW`, `healpy`, `camb`/`class`) |

![Diagram: Python environments, package managers and Jupyter kernels](../book/diagrams/python-environments.svg)

**Kernel $\neq$ terminal.** A Jupyter kernel executes inside one environment's `sys.executable`, whereas a new terminal shell does **not** activate that environment automatically — activate `.venv` first (or use `uv run`), run `python -m pip` rather than bare `pip`, and if mixing in Conda, install `conda-forge` packages *before* `python -m pip`. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/python-environments.dot).

**Scientific Python stack:** `numpy` provides typed $n$-D arrays and vectorised maths; `scipy` adds numerical integration, optimisation/fitting and transforms; `matplotlib` builds publication figures; and `astropy` supplies physical units, constants, FITS/WCS I/O and `QTable`. Astronomy workflows prefer `QTable` over `pandas.DataFrame` because `QTable` preserves physical `Quantity` column units, multi-D spectral or posterior columns, and FITS/ECSV headers (with `.to_pandas()` available when flat export is needed).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable

# Locate the course root directory (no need to edit)
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / 'jupyter_data_languages']
for _candidate in _candidates:
    if (_candidate / 'course2026/__init__.py').is_file() and (_candidate / 'data/mock_galaxies.ecsv').is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({
    'figure.figsize': (7, 4),
    'font.size': 11,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
})
print('Interpreter:', sys.executable, '| NumPy:', np.__version__)

## Inspect the catalogue

Astropy's `QTable` stores tabular columns as unit-aware `Quantity` arrays so physical dimensions travel with the numbers. We load `mock_galaxies.ecsv`, an **Extended CSV (`.ecsv`)** file: plain-text CSV preceded by a `#`-commented YAML header that records column units, data types and provenance metadata — convenient for sharing and version-controlling both observational sub-catalogues and synthetic simulation tables.

Before running the next cell, predict what columns, units and metadata dictionary (`table.meta`) a survey or simulation table needs to record.

In [ ]:
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
print('Rows:', len(table), '| metadata:', table.meta)
table[:3].pprint_all()

## Arrays, shapes and masks

Never apply separate row filters to individual columns: build **one shared boolean mask**, inspect how many rows it removes (`~valid` is boolean NOT), and apply it to every column so objects remain aligned. Calling `.to_value(u.Mpc)` converts compatible Astropy units and extracts a plain `float` NumPy array. Because observed flux dims with distance as $F \propto L / (4\pi d^2)$, astronomy's logarithmic `apparent_magnitude` scale runs backwards: **larger numbers mean fainter (typically more distant) galaxies**.

![Diagram: One mask preserves row alignment](../book/diagrams/shared-mask.svg)

**Keep objects aligned.** Filtering columns separately can produce matching array lengths while silently pairing one galaxy's distance with another's velocity. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/shared-mask.dot).

In [ ]:
distance = table['distance'].to_value(u.Mpc)
velocity = table['velocity'].to_value(u.km / u.s)
sigma = table['velocity_error'].to_value(u.km / u.s)
magnitude = np.asarray(table['apparent_magnitude'])

valid = (
    np.isfinite(distance) & np.isfinite(velocity) & np.isfinite(sigma)
    & np.isfinite(magnitude) & (distance > 0) & (sigma > 0)
)
print('Shape:', distance.shape, '| usable:', valid.sum(), '| removed:', (~valid).sum())
assert valid.sum() == 395
assert distance[valid].shape == velocity[valid].shape == sigma[valid].shape

### Task A

1. Count missing (`~np.isfinite`) velocities and nonpositive (`<= 0` or non-finite) velocity errors separately, and inspect the excluded rows with `table[~valid]`.
2. Explain why replacing missing velocities with `0.0` would corrupt a mean or linear fit *(hint: `np.count_nonzero(condition)` counts `True` entries).*

In [ ]:
# Task A: count missing velocities and nonpositive errors separately

### Shapes and broadcasting

NumPy aligns array shapes **from right to left (trailing axes first)**: two dimensions are compatible only when they are equal or one of them is `1`. Multiplying an `(N, 3)` multi-band table by a 1D per-object array of shape `(N,)` fails because trailing axis `3 != N`, whereas indexing with `[:, None]` reshapes `(N,)` to `(N, 1)` so one factor scales each object's three bands cleanly. In 2D reductions, **the `axis` argument names the dimension being collapsed**: `axis=0` collapses rows into per-band summaries `(3,)`, whereas `axis=1` collapses columns into per-galaxy totals `(N,)`.

![Diagram: Broadcast one scale per object](../book/diagrams/broadcasting.svg)

**Scale rows, not columns.** Reshaping `(N,)` to `(N, 1)` stretches each object's correction factor across its three band columns. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/broadcasting.dot).

In [ ]:
# Synthetic 3-band fluxes (g, r, i) for the 395 usable galaxies: shape (395, 3)
flux_bands = (1e4 / distance[valid, None] ** 2) * np.array([0.8, 1.0, 1.3])
extinction_corr = np.linspace(0.9, 1.1, valid.sum())  # shape (395,)
corrected_flux = flux_bands * extinction_corr[:, None]

print('3-band matrix:', flux_bands.shape, '| row factor:', extinction_corr[:, None].shape)
print('Per-band mean (axis=0):', corrected_flux.mean(axis=0).shape, '| per-galaxy sum (axis=1):', corrected_flux.sum(axis=1).shape)
assert corrected_flux.shape == (395, 3) and corrected_flux.mean(axis=0).shape == (3,) and corrected_flux.sum(axis=1).shape == (395,)
np.testing.assert_allclose(corrected_flux[-1], flux_bands[-1] * 1.1)

## Units, constants and FITS

Never infer units from the size of a number: rely on documented headers, physical constants and checked conversions. `astropy.constants` provides CODATA/IAU constants carrying full dimensions and provenance (preventing CGS-versus-SI mix-ups in theoretical calculations), while `equivalencies` (`u.spectral()`, `u.doppler_optical`) translate spectral wavelengths into frequencies or recession velocities — and incompatible conversions raise `UnitConversionError` immediately.

In [ ]:
from astropy import constants as const

r_s = (2 * const.G * const.M_sun / const.c**2).to(u.km)
obs_ha, rest_ha = 660.0 * u.nm, 656.28 * u.nm
freq_ha = obs_ha.to(u.THz, equivalencies=u.spectral())
v_opt = obs_ha.to(u.km / u.s, equivalencies=u.doppler_optical(rest_ha))
print(f'Solar r_s: {r_s:.2f} | H-alpha {obs_ha} -> {freq_ha:.1f} -> v_opt = {v_opt:.1f}')

velocity_ms = table['velocity'].to(u.m / u.s)
np.testing.assert_allclose(velocity_ms.value[valid], 1000 * velocity[valid])
try:
    table['distance'].to(u.s)
except u.UnitConversionError:
    print('Expected: a distance cannot be converted to seconds')

Large astronomical catalogues and images are commonly archived in **FITS (Flexible Image Transport System)**, a binary format organised into numbered Header/Data Units (HDUs). Writing a table creates an empty Primary Header (`HDU 0`) followed by a Binary Table (`HDU 1`) whose `TUNITn` header cards preserve physical column units. Because standard FITS headers enforce strict 8-character keyword rules and 80-character card limits — unlike ECSV's free-form YAML header — we clear long free-text entries from `subset.meta` before writing.

![Diagram: ECSV text metadata vs FITS Header and Data Units](../book/diagrams/fits-vs-ecsv.svg)

**Text ECSV vs binary FITS.** Both formats preserve column units (`TUNITn` in FITS), while ECSV accommodates richer free-form dictionary metadata. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/fits-vs-ecsv.dot).

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'mock_subset.fits'
    subset = table[['distance', 'velocity', 'velocity_error']][valid].copy()
    subset.meta.clear()
    subset.write(path)
    restored = QTable.read(path)
    np.testing.assert_allclose(restored['distance'].to_value(u.Mpc), distance[valid])
    print('FITS round-trip rows:', len(restored), '| unit:', restored['distance'].unit)

### 2D FITS images, WCS and colormaps

FITS `PrimaryHDU` and `ImageHDU` blocks store 2D sky cutouts or simulation slices alongside World Coordinate System (`WCS`) header keywords mapping pixel `(x, y)` to sky or physical coordinates (`RA`, `Dec`). Follow three essential 2D display rules:
1. Pass `origin='lower'` so pixel `(0, 0)` sits at the bottom-left rather than top-down matrix order.
2. Apply an astronomical stretch (`AsinhStretch` or `LogStretch` via `ImageNormalize`) so bright cores do not wash out faint outer structure.
3. Choose **perceptually uniform sequential** colormaps (`magma`, `viridis`) for positive intensity or density, **zero-centred diverging** colormaps (`RdBu_r` with `CenteredNorm(vcenter=0)`) for signed residuals or velocity fields, and avoid rainbow/`jet`.

In [ ]:
from matplotlib.colors import CenteredNorm
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval
from astropy.wcs import WCS

rng = np.random.default_rng(7)
yy, xx = np.mgrid[:64, :64]
smooth_model = (
    480.0 * np.exp(-0.5 * (((xx - 31.5) / 3.2) ** 2 + ((yy - 31.5) / 2.4) ** 2))
    + 36.0 * np.exp(-0.5 * (((xx - 31.5) / 13.0) ** 2 + ((yy - 31.5) / 9.5) ** 2))
)
clump = 32.0 * np.exp(-0.5 * (((xx - 45.0) / 2.2) ** 2 + ((yy - 22.0) / 2.2) ** 2))
cutout = smooth_model + clump + rng.normal(0.0, 2.5, size=(64, 64))

wcs = WCS(naxis=2)
wcs.wcs.crpix, wcs.wcs.cdelt = [32.5, 32.5], [-0.00028, 0.00028]
wcs.wcs.crval, wcs.wcs.ctype = [150.119, 2.205], ['RA---TAN', 'DEC--TAN']
hdu = fits.PrimaryHDU(data=cutout, header=wcs.to_header())

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), subplot_kw={'projection': WCS(hdu.header)})
norm_img = ImageNormalize(hdu.data, interval=PercentileInterval(99.5), stretch=AsinhStretch())
im0 = axes[0].imshow(hdu.data, origin='lower', cmap='magma', norm=norm_img)
fig.colorbar(im0, ax=axes[0], pad=0.03, fraction=0.046, label='Flux [arb.]')
axes[0].set(xlabel='RA', ylabel='Dec', title='FITS cutout (asinh stretch)')

im1 = axes[1].imshow(hdu.data - smooth_model, origin='lower', cmap='RdBu_r', norm=CenteredNorm(vcenter=0.0))
fig.colorbar(im1, ax=axes[1], pad=0.03, fraction=0.046, label='Residual')
axes[1].set(xlabel='RA', ylabel='Dec', title='Data - smooth model (zero-centred)')
for ax in axes:
    ax.coords[0].set_major_formatter('d.ddd')
    ax.coords[1].set_major_formatter('d.ddd')
fig.subplots_adjust(wspace=0.44, left=0.08, right=0.92, bottom=0.14, top=0.88)
plt.show()

### Task B

Explain to a partner which provenance metadata `subset.meta.clear()` discarded, and why `restored['distance'].to_value(u.Mpc)` is safer than `.value` if a collaborator saved distances in `kpc`. Write one `assert` check you would run immediately after reading an external FITS catalogue.

**Break** *(after completing Task B).*

In [ ]:
# Task B: write one assertion you would run after reading a FITS table

## Plot the data and its uncertainty

Build explicit `fig, axes = plt.subplots(...)` objects rather than stateful `plt.*` calls so every curve, axis scale and legend targets a specific panel unambiguously. Pair discrete `ax.errorbar` points (subsampled `[::8]` here so individual $1\sigma$ velocity bars stay legible; $d$ carries no horizontal bars because input distances are exact) with continuous `ax.fill_between` model/uncertainty bands, and call `ax.invert_yaxis()` on magnitude axes so brighter objects sit higher. Save vector `.pdf` via `fig.savefig(...)` for LaTeX papers and `dpi=300` `.png` for slides — adding `rasterized=True` or `ax.hexbin` when plotting $\ge 10^5$ survey sources or MCMC samples so vector PDFs remain lightweight.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
d_grid = np.linspace(20, 200, 200)
v_line = 70.0 * d_grid + 300.0
med_sigma = np.median(sigma[valid])
indices = np.flatnonzero(valid)[::8]

axes[0].fill_between(
    d_grid, v_line - med_sigma, v_line + med_sigma,
    color='C0', alpha=0.18, label=f'Injected line ± median σ ({med_sigma:.0f} km/s)',
)
axes[0].plot(d_grid, v_line, color='C0', lw=1.6)
axes[0].errorbar(
    distance[indices], velocity[indices], yerr=sigma[indices],
    fmt='o', color='0.15', ms=3.5, elinewidth=1.0, capsize=2,
    label=f'Every 8th usable galaxy (n={len(indices)})',
)
axes[0].set(
    xlabel='Distance (exact input) [Mpc]',
    ylabel='Velocity [km/s]',
    title='Velocity relation and 1σ errors',
)
axes[0].legend(fontsize=8.5, loc='upper left')

bright = valid & (magnitude < 16.5)
faint = valid & (magnitude >= 16.5)
axes[1].scatter(distance[faint], magnitude[faint], s=14, color='0.75', alpha=0.8, label=f'Excluded m >= 16.5 (n={faint.sum()})')
axes[1].scatter(distance[bright], magnitude[bright], s=14, color='C0', alpha=0.8, label=f'Selected m < 16.5 (n={bright.sum()})')
axes[1].axhline(16.5, color='C3', ls='--', lw=1.4, label='Flux limit m = 16.5')
axes[1].invert_yaxis()
axes[1].set(
    xlabel='Distance (exact input) [Mpc]',
    ylabel='Apparent magnitude [mag]',
    title='Flux limit truncates distant objects',
)
axes[1].legend(fontsize=8.5, loc='lower left')
fig.tight_layout()

with tempfile.TemporaryDirectory() as tmp:
    pdf_path, png_path = Path(tmp) / 'catalogue_summary.pdf', Path(tmp) / 'catalogue_summary.png'
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=300)
    assert pdf_path.stat().st_size > 0 and png_path.stat().st_size > 0
plt.show()

### Your turn

Plot the velocity residuals `resid = velocity[valid] - (70.0 * distance[valid] + 300.0)` against `distance[valid]` (either subsampling `[::8]` with `yerr=sigma[valid][::8]` or colouring all 395 points by `magnitude[valid]` with `fig.colorbar`), add a horizontal `ax.axhline(0.0, ls='--')` reference line, and check whether the residuals show any systematic trend with distance.

In [ ]:
# Your turn: plot velocity residuals against distance with ax.axhline(0.0, ls='--')

## Investigate a survey selection

Keep data-quality filtering (`valid`, 395 rows) distinct from physical or observational selection (`apparent_magnitude < 16.5`, 262 rows), and compare populations on **identical histogram bin edges**. Raw counts show where objects drop out; normalised probability densities compare distribution shapes.

In **observational** terms, `m < 16.5` is a flux-limited cut that preferentially loses distant galaxies (and in surveys with distance scatter, produces classical Malmquist bias). In **statistical/theoretical** terms, conditioning on $m < 16.5$ truncates the marginal distance distribution $p(d)$ — shifting $\bar{d}$ from $110.7\text{ Mpc}$ to $87.5\text{ Mpc}$ — while leaving the conditional relation $E[v \mid d]$ unbiased here because $d$ is exact and velocity noise is independent of $m$.

![Diagram: Filtering and survey selection are separate decisions](../book/diagrams/catalogue-selection.svg)

**Data hygiene vs survey cut.** Changing `magnitude_limit` alters the selected sample (262 of 395 at `m < 16.5`) without changing which 395 rows are valid. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/catalogue-selection.dot).

In [ ]:
magnitude_limit = 16.5
selected = valid & (magnitude < magnitude_limit)
print('Selected:', selected.sum(), 'of', valid.sum(), 'usable rows')
assert selected.sum() == 262

bins = np.linspace(20, 200, 13)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = ['Counts per bin', 'Normalised probability density']
for ax, density, title in zip(axes, [False, True], titles):
    ax.hist(distance[valid], bins=bins, histtype='step', lw=2, density=density,
            label='All usable (n=395)')
    ax.hist(distance[selected], bins=bins, histtype='step', lw=2, density=density,
            label=f'm < {magnitude_limit} (n={selected.sum()})')
    ax.set(
        title=title,
        xlabel='Distance [Mpc]',
        ylabel='Probability density [1/Mpc]' if density else 'Objects per bin',
    )
axes[0].legend(fontsize=8.5)
fig.tight_layout()
plt.show()
print('Mean distances [Mpc]:', distance[valid].mean().round(1), distance[selected].mean().round(1))

### Task C

1. Using a separate variable `test_limit`, predict whether tightening the cut to `15.5` or relaxing it to `17.5` lowers or raises the selected count and mean distance, then verify both numerically.
2. Draft a three-sentence caption stating what is plotted, what rows were excluded, and why shifting $p(d)$ does **not** automatically bias $v(d)$ under this mock's noise model — then name one realistic effect that *would* bias the fit.

In [ ]:
# Task C: check counts and mean distances for test_limit = 15.5 and 17.5 (leave `selected` unchanged)
test_limit = 15.5

## Exit check

Before moving on, verify you can explain: why a notebook kernel differs from a fresh shell, why 395 of 400 rows are usable, why `(N, 3) * (N, 1)` broadcasts per row while `axis=0` vs `axis=1` collapses rows vs columns, why units and constants come from headers/libraries rather than guesswork, why 2D cutouts use `origin='lower'` and zero-centred residual colormaps, and why a flux limit changes $p(d)$ without changing data validity. Run **Restart Kernel and Run All Cells** to confirm clean top-to-bottom execution, then compare with [solutions](../book/solutions.md).